In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
def generate_heatmap(data, handle_nan='drop', impute_value=None, drop_thresh=0.5):
    """
    Generates a heatmap from the given DataFrame.

    Parameters:
    data (pd.DataFrame): The input DataFrame.
    handle_nan (str): How to handle NaN values. Options are 'drop', 'impute', or 'ignore'.
    impute_value (scalar, dict, or None): Value to impute NaNs with, if handle_nan is 'impute'. 
                                          Can be a scalar or a dict specifying which value to use for each column.
    drop_thresh (float): The threshold for dropping columns. Columns with more than this fraction of NaNs will be dropped.

    Returns:
    None
    """

    # Drop columns containing 'id'
    cols_to_drop = [col for col in data.columns if 'id' in col.lower()]
    data = data.drop(columns=cols_to_drop)

    # Convert datetime columns to integers
    for column in data.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]', 'timedelta64[ns]']).columns:
        data[column] = data[column].astype('int64')

    # Convert specific categorical variable 'first_careunit' to integer
    if 'first_careunit' in data.columns:
        data['first_careunit'], _ = pd.factorize(data['first_careunit'])

    # Convert remaining non-numeric columns to numeric where applicable
    for column in data.select_dtypes(include=['object', 'category']).columns:
        if column != 'first_careunit':  # Skip 'first_careunit' since it's already converted
            data[column], _ = pd.factorize(data[column])

    # Handle NaN values
    if handle_nan == 'drop':
        data = data.dropna(axis=1, thresh=int(drop_thresh * len(data)))
        data = data.dropna(axis=0)
    elif handle_nan == 'impute':
        if impute_value is not None:
            data = data.fillna(impute_value)
        else:
            data = data.fillna(data.mean())

    # Generate the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.show()
